In [ ]:
from groundingdino.util.inference import load_model, load_image, predict, annotate
from pathlib import Path
import torch
import json

# Category name conversion dictionary
CATEGORY_CONVERSION = {
    'noise': 'noise',
    'animal': 'animal',
    'human.pedestrian.adult': 'person',
    'human.pedestrian.child': 'person',
    'human.pedestrian.construction_worker': 'person',
    'human.pedestrian.personal_mobility': 'person',
    'human.pedestrian.police_officer': 'person',
    'human.pedestrian.stroller': 'person',
    'human.pedestrian.wheelchair': 'person',
    'movable_object.barrier': 'barrier',
    'movable_object.debris': 'debris',
    'movable_object.pushable_pullable': 'pushable_pullable',
    'movable_object.trafficcone': 'trafficcone',
    'static_object.bicycle_rack': 'bicycle_rack',
    'vehicle.bicycle': 'bicycle',
    'vehicle.bus.bendy': 'bus',
    'vehicle.bus.rigid': 'bus',
    'vehicle.car': 'car',
    'vehicle.construction': 'construction_vehicle',
    'vehicle.emergency.ambulance': 'ambulance',
    'vehicle.emergency.police': 'police_car',
    'vehicle.motorcycle': 'motorcycle',
    'vehicle.trailer': 'trailer',
    'vehicle.truck': 'truck',
    'flat.driveable_surface': 'driveable_surface',
    'flat.other': 'other',
    'flat.sidewalk': 'sidewalk',
    'flat.terrain': 'terrain',
    'static.manmade': 'manmade',
    'static.other': 'other',
    'static.vegetation': 'vegetation',
    'vehicle.ego': 'car'
}
# Resolve paths relative to this notebook directory
GROUNDING_DINO_ROOT = Path.cwd().parent / "GroundingDINO"
CONFIG_PATH = GROUNDING_DINO_ROOT / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
WEIGHTS_PATH = GROUNDING_DINO_ROOT / "weights/groundingdino_swinb_cogcoor.pth"
IMAGE_PATH = GROUNDING_DINO_ROOT / ".asset/cat_dog.jpeg"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Create the model and load the weights
model = load_model(str(CONFIG_PATH), str(WEIGHTS_PATH), device=device)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-mini"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "category.json") as f:
    categories = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
# Create hash maps for token lookup
category_lookup = {category["name"]: category["token"] for category in categories}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors}
print(f"category_names: {[category['name'] for category in categories]}")
print(f"sensor_channels: {[sensor['channel'] for sensor in sensors]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")
# Validate the category conversion dictionary
for category in CATEGORY_CONVERSION.keys():
    if category not in category_lookup:
        raise ValueError(f"Category '{category}' not found in nuScenes categories.")

In [ ]:
# Select the scenes and camera channel
SCENE_NAME = "scene-0061"
CAMERA_CHANNEL = "CAM_FRONT"
scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
samples = [sample for sample in samples_all if sample["scene_token"] == scene["token"]]
print(f"num_samples: {len(samples)}")
# Select the samples, sample_data, sample_annotation, and instances in the scene
samples = [sample for sample in samples_all if sample["scene_token"] == scene["token"]]
sample_tokens = set(sample["token"] for sample in samples)
sample_data = [sd for sd in sample_data_all if sd["sample_token"] in sample_tokens]
sample_annotations = [sa for sa in sample_annotations_all if sa["sample_token"] in sample_tokens]
instance_tokens = set(sa["instance_token"] for sa in sample_annotations)
instances = [inst for inst in instances_all if inst["token"] in instance_tokens]